In [ ]:
#Running HDBSCAN in addition to use of neural networks ( DO NOT RUN IT TAKES 15mins)
import os
import pandas as pd
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
import hdbscan
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
import umap

# ===============================
# 1. Load cleaned data
# ===============================
df = pd.read_csv("../data/Reclamations_clean.csv")

TEXT_COL = "texte_reclamation"

# Truncate text to speed up embeddings
MAX_CHARS = 300
texts = df[TEXT_COL].astype(str).str[:MAX_CHARS].tolist()

print(f"Number of complaints: {len(texts)}")

# ===============================
# 2. Check GPU availability
# ===============================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ===============================
# 3. Load embedding model
# ===============================
model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2",
    device=device
)

# ===============================
# 4. Compute or load embeddings (CACHE)
# ===============================
EMB_PATH = "../data/complaint_embeddings.npy"

if os.path.exists(EMB_PATH):
    print("Loading cached embeddings...")
    embeddings = np.load(EMB_PATH)
else:
    print("Computing embeddings...")
    embeddings = model.encode(
        texts,
        batch_size=64 if device == "cuda" else 32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    np.save(EMB_PATH, embeddings)

# Normalize embeddings for cosine similarity
embeddings = normalize(embeddings)

# ===============================
# 5. UMAP dimensionality reduction
# ===============================
umap_reducer = umap.UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

embeddings_umap = umap_reducer.fit_transform(embeddings)

# ===============================
# 6. HDBSCAN clustering
# ===============================
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom"
)

labels = clusterer.fit_predict(embeddings_umap)
df["cluster"] = labels

# ===============================
# 7. Cluster statistics
# ===============================
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_ratio = np.mean(labels == -1)

print(f"\nNumber of clusters found: {n_clusters}")
print(f"Noise ratio: {noise_ratio:.2%}")

# ===============================
# 8. Silhouette score (exclude noise)
# ===============================
mask = labels != -1

if len(set(labels[mask])) > 1:
    sil_score = silhouette_score(
        embeddings_umap[mask],
        labels[mask],
        metric="euclidean"
    )
    print(f"Silhouette coefficient (no noise): {sil_score:.4f}")
else:
    print("Silhouette score cannot be computed (only one cluster).")

# ===============================
# 9. Inspect cluster examples
# ===============================
def show_cluster_examples(df, cluster_id, n=5):
    print(f"\n--- Cluster {cluster_id} ---")
    samples = df[df["cluster"] == cluster_id][TEXT_COL].head(n)
    for i, txt in enumerate(samples, 1):
        print(f"{i}. {txt}")

for cid in sorted(c for c in df["cluster"].unique() if c != -1):
    show_cluster_examples(df, cid)

# ===============================
# 10. Save clustered data
# ===============================
OUTPUT_PATH = "../data/Reclamations_clustered.csv"
df.to_csv(OUTPUT_PATH, index=False)

print(f"\nClustered data saved to: {OUTPUT_PATH}")


Number of complaints: 100000
CUDA available: False
Using device: cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
e:\Anaconda\envs\DM_ENV\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wassi\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate devel

Computing embeddings...


Batches: 100%|██████████| 3125/3125 [05:20<00:00,  9.75it/s]
e:\Anaconda\envs\DM_ENV\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Number of clusters found: 51
Noise ratio: 0.00%
Silhouette coefficient (no noise): 0.9680

--- Cluster 0 ---
1. Client signale problème lors de la résiliation, facturation continue.
2. Client signale problème lors de la résiliation, facturation continue.
3. Client signale problème lors de la résiliation, facturation continue.
4. Client signale problème lors de la résiliation, facturation continue.
5. Client signale problème lors de la résiliation, facturation continue.

--- Cluster 1 ---
1. Abonné rapporte image figée sur certaines chaines.
2. Abonné rapporte image figée sur certaines chaines.
3. Abonné rapporte image figée sur certaines chaines.
4. Abonné rapporte image figée sur certaines chaines.
5. Abonné rapporte image figée sur certaines chaines.

--- Cluster 2 ---
1. Abonné rapporte bruit important sur la ligne téléphonique.
2. Abonné rapporte bruit important sur la ligne téléphonique.
3. Abonné rapporte bruit important sur la ligne téléphonique.
4. Abonné rapporte bruit import